# Superfermion Framework Test

Exercises the core API surface of Superfermion as a quantum computation framework.

In [ ]:
import superfermion as sf
print(f"Superfermion v{sf.__version__}")

## 1. Circuit Construction

Build circuits with the fluent API. All standard gates are supported.

In [ ]:
# Bell state: (|00> + |11>) / sqrt(2)
bell = sf.Circuit(2).h(0).cnot(0, 1)
print(f"Bell circuit: {bell.n_qubits} qubits, depth {bell.depth}, {bell.gate_count} gates")
print(f"Gate list: {bell.to_gate_list()}")

In [ ]:
# GHZ state: (|000...0> + |111...1>) / sqrt(2)
n = 5
ghz = sf.Circuit(n).h(0)
for i in range(n - 1):
    ghz.cnot(i, i + 1)
print(f"GHZ-{n}: depth {ghz.depth}, {ghz.gate_count} gates")
print(f"Gate list: {ghz.to_gate_list()}")

In [ ]:
# Parametric circuit with symbolic parameters
theta = sf.param("theta")
phi = sf.param("phi")
parametric = sf.Circuit(2).rx(theta, 0).ry(phi, 1).cnot(0, 1)
print(f"Parameters: {parametric.parameters}")
print(f"Unbound count: {parametric.n_parameters}")

bound = parametric.bind({"theta": 1.57, "phi": 0.785})
print(f"After bind: {bound.n_parameters} unbound parameters")

## 2. Execution via sf.run()

In [ ]:
result = sf.run(bell, device="cpu", shots=4096)
print(f"Counts: {result.counts}")
print(f"Statevector: {result.statevector}")
print(f"Probabilities: {result.get_probabilities()}")

In [ ]:
ghz_result = sf.run(ghz, device="cpu", shots=2048)
print("GHZ-5 counts (should only have 00000 and 11111):")
for bitstring, count in sorted(ghz_result.counts.items(), key=lambda x: -x[1]):
    print(f"  |{bitstring}> : {count}")

In [ ]:
param_result = sf.run(bound, device="cpu", shots=1000)
print(f"Parametric circuit counts: {param_result.counts}")

## 3. Multiple Backends

In [ ]:
backends = sf.list_backends()
print(f"Available backends ({len(backends)}): {backends}")

In [ ]:
test_circuit = sf.Circuit(3).h(0).cnot(0, 1).cnot(1, 2)

for backend_name in ["statevector", "numpy"]:
    try:
        r = sf.run(test_circuit, device=backend_name, shots=1000)
        top = sorted(r.counts.items(), key=lambda x: -x[1])[:3]
        print(f"{backend_name:15s} -> {dict(top)}")
    except Exception as e:
        print(f"{backend_name:15s} -> {type(e).__name__}: {e}")

## 4. Observables & Expectation Values

In [ ]:
from superfermion.observables.core import PauliString, SparsePauliOp, expval

# SparsePauliOp takes list of pauli strings and coefficients
hamiltonian = SparsePauliOp(["ZI", "XX"], coeffs=[1.0, 0.5])
print(f"Hamiltonian: {hamiltonian}")
print(f"Number of terms: {len(hamiltonian.terms)}")

bell_sv = sf.run(bell, device="cpu", shots=0)
ev = expval(bell_sv.statevector, hamiltonian)
print(f"<Bell|H|Bell> = {ev:.6f}")

## 5. Experiment Tracking

In [ ]:
tracker = sf.LocalTracker(name="notebook-test")

with sf.experiment("bell-experiment", tracker=tracker):
    r1 = sf.run(bell, device="cpu", shots=1000)
    r2 = sf.run(ghz, device="cpu", shots=1000)

print(f"Tracker recorded {len(tracker.runs)} run(s)")
for i, run_data in enumerate(tracker.runs):
    print(f"  Run {i+1}: {run_data['n_qubits']}q, {run_data['gate_count']} gates, "
          f"{run_data['shots']} shots, {run_data['n_outcomes']} outcomes")

## 6. Device Protocol

In [ ]:
from superfermion.devices import DeviceExecutor, DeviceCapabilities
from superfermion.devices.local import LocalDevice

local = LocalDevice("statevector")
caps = local.capabilities()
print(f"Device: {local}")
print(f"Max qubits: {caps.max_qubits}")
print(f"Native gates: {caps.native_gates[:10]}...")
print(f"Is simulator: {caps.is_simulator}")

# Execute directly via the device protocol
result = local.execute(bell, shots=500)
print(f"\nDirect execution counts: {result.counts}")

## 7. Serialization Round-trip

In [ ]:
from superfermion.serialization import save_circuit, load_circuit, to_qasm3
import tempfile, os

# JSON round-trip
with tempfile.NamedTemporaryFile(suffix=".sfc", delete=False) as f:
    path = f.name
save_circuit(bell, path)
loaded = load_circuit(path)
print(f"JSON round-trip: {loaded.n_qubits} qubits, {loaded.gate_count} gates")
r_loaded = sf.run(loaded, device="cpu", shots=1000)
print(f"Loaded circuit counts: {r_loaded.counts}")
os.unlink(path)

# QASM3
qasm_str = to_qasm3(bell)
print(f"\nQASM3 output:\n{qasm_str}")

## 8. Compiler

In [ ]:
redundant = sf.Circuit(2)
redundant.h(0).h(0)  # H*H = I, should cancel
redundant.cnot(0, 1)
redundant.x(1).x(1)  # X*X = I, should cancel

print(f"Before compile: {redundant.gate_count} gates -> {redundant.to_gate_list()}")
compiled = sf.compile(redundant)
print(f"After compile:  {compiled.gate_count} gates -> {compiled.to_gate_list()}")

## 9. Results Introspection

In [ ]:
result = sf.run(bell, device="cpu", shots=10000)

print(f"Total shots: {sum(result.counts.values())}")
print(f"Statevector: {result.statevector}")
print(f"Probabilities: {result.get_probabilities()}")
print(f"Most probable: |{max(result.counts, key=result.counts.get)}>")

# Serialization round-trip
d = result.to_dict()
restored = sf.RunResult.from_dict(d)
print(f"\nResult round-trip counts match: {restored.counts == result.counts}")

## 10. Larger Circuit — QFT

In [ ]:
import math

def qft(n: int) -> sf.Circuit:
    """Build an n-qubit Quantum Fourier Transform circuit."""
    c = sf.Circuit(n)
    for i in range(n):
        c.h(i)
        for j in range(i + 1, n):
            angle = math.pi / (2 ** (j - i))
            c.cp(angle, i, j)
    # Swap qubits for standard QFT ordering
    for i in range(n // 2):
        c.swap(i, n - 1 - i)
    return c

qft_circuit = qft(6)
print(f"QFT-6: {qft_circuit.gate_count} gates, depth {qft_circuit.depth}")

qft_result = sf.run(qft_circuit, device="cpu", shots=2048)
top_5 = sorted(qft_result.counts.items(), key=lambda x: -x[1])[:5]
print(f"Top 5 outcomes: {dict(top_5)}")
print(f"Unique outcomes: {len(qft_result.counts)}")

## Summary

Core framework APIs exercised:

| # | Feature | API |
|---|---------|-----|
| 1 | Circuit construction | `sf.Circuit`, fluent gate methods |
| 2 | Symbolic parameters | `sf.param()`, `circuit.bind()` |
| 3 | Execution | `sf.run(circuit, device, shots)` |
| 4 | Multiple backends | `sf.list_backends()` |
| 5 | Observables | `PauliString`, `SparsePauliOp`, `expval` |
| 6 | Experiment tracking | `sf.experiment()`, `LocalTracker` |
| 7 | Device protocol | `DeviceExecutor`, `LocalDevice` |
| 8 | Serialization | `save_circuit/load_circuit`, `to_qasm3` |
| 9 | Compilation | `sf.compile()` (gate cancellation) |
| 10 | Result introspection | `RunResult.to_dict/from_dict` |